# 090-CS-ood-rdmfsによる学認RDMストレージマウント機能の追加

このnotebookでは、Open OnDemandのsys appとして [CS-ood-rdmfs](https://github.com/RCOSDP/CS-ood-rdmfs) を追加し、学認RDMのストレージをOpen OnDemandホストにマウントできるようにするための準備を行います。

## 前提

* 020-フロントエンドのセットアップのnotebookにより、Open OnDemandのフロントエンド設定が完了していること。
* 010-インストールと同じディレクトリで実行し、カレントディレクトリに `group_vars`、`inventory.yml`、`ansible.cfg` が存在すること。
* 学認RDMのAPI URL や Token の発行自体はこのnotebookの対象外。


## 概要

このnotebookでは以下の2つのリポジトリを組み込みます。

* [CS-rdmfs](https://github.com/RCOSDP/CS-rdmfs) をマスターノードに導入する。
* [CS-ood-rdmfs](https://github.com/RCOSDP/CS-ood-rdmfs) を `/var/www/ood/apps/sys` 配下に配置する。

[CS-ood-rdmfs](https://github.com/RCOSDP/CS-ood-rdmfs) は Open OnDemand から mount / unmount を操作するための sys app であり、実際の mount 処理は [CS-rdmfs](https://github.com/RCOSDP/CS-rdmfs) の `bin/start.sh` を `/usr/local/sbin/rdmfs_mount.sh` として配置して呼び出します。


[CS-ood-rdmfs](https://github.com/RCOSDP/CS-ood-rdmfs) の README では、appディレクトリと `tmp` に `chmod 777` を付与する手順が案内されています。本notebookも同じ構成を採用します。運用環境に適用する際には、必要に応じて権限設計を見直してください。

## 準備

### group_varsの読み込み

group_varsの読み込みに先立ち、ユニットグループ名をチェックします。

In [ ]:
!ls -1 group_vars/*.yml | sed -e 's/^group_vars\///' -e 's/\.yml//' | sort

`ugroup_name`にユニットグループ名を設定します。

In [ ]:
ugroup_name = ''

group_varsを読み込み、ansibleで利用するノード集合を設定します。

In [ ]:
%run scripts/group.py

gvars = load_group_vars(ugroup_name)
target_master = f'{ugroup_name}_master'

ansibleでマスターノードにアクセスできることを確認します。

In [ ]:
!ansible -m ping {target_master}

## 導入パラメータ

導入時に利用するリポジトリ、ブランチ、配置先を指定します。

In [ ]:
cs_rdmfs_repo = 'https://github.com/RCOSDP/CS-rdmfs.git'
cs_rdmfs_ref = 'main'

cs_ood_rdmfs_repo = 'https://github.com/RCOSDP/CS-ood-rdmfs.git'
cs_ood_rdmfs_ref = 'master'

install_root = '/opt/CS-rdmfs'
wrapper_path = '/usr/local/sbin/rdmfs_mount.sh'
ood_sys_root = '/var/www/ood/apps/sys'
ood_app_root = f'{ood_sys_root}/CS-ood-rdmfs'

## ホスト前提の設定

### 必要パッケージの導入

[FUSE](https://github.com/libfuse/libfuse) と [CS-rdmfs](https://github.com/RCOSDP/CS-rdmfs) の導入に必要なパッケージをマスターノードに導入します。[CS-rdmfs](https://github.com/RCOSDP/CS-rdmfs)の要件に従いPython 3.11 関連パッケージも導入します。

In [ ]:
!ansible {target_master} -b -m dnf -a \
    'name=fuse3,fuse3-devel,git,attr,python3-pip,python311,python3.11-devel state=present'

### FUSE の設定

[CS-rdmfs](https://github.com/RCOSDP/CS-rdmfs) は `allow_other` を利用してマウントするため、`/etc/fuse.conf` で `user_allow_other` を有効にします。

In [ ]:
!ansible {target_master} -b -m replace -a \
    "path=/etc/fuse.conf \
     regexp='^# user_allow_other' \
     replace='user_allow_other'"

In [ ]:
!ansible {target_master} -b -a \
    "grep -n '^user_allow_other' /etc/fuse.conf"

## [CS-rdmfs](https://github.com/RCOSDP/CS-rdmfs) 実行環境の構築

[CS-rdmfs](https://github.com/RCOSDP/CS-rdmfs) のリポジトリを配置します。既に配置済みの場合は指定したrefに更新します。

In [ ]:
!ansible {target_master} -b -m git -a \
    'repo={cs_rdmfs_repo} dest={install_root} version={cs_rdmfs_ref} update=yes force=yes'

Python 3.11 の virtualenv を作成します。

In [ ]:
!ansible {target_master} -b -m shell -a \
    'python3.11 -m venv {install_root}/venv'

pipを更新し、requirementsを導入します。

In [ ]:
!ansible {target_master} -b -m pip -a \
    'name=pip state=latest virtualenv={install_root}/venv'
!ansible {target_master} -b -m pip -a \
    'requirements={install_root}/requirements.txt \
     virtualenv={install_root}/venv'

[CS-rdmfs](https://github.com/RCOSDP/CS-rdmfs) 本体をインストールします。

In [ ]:
!ansible {target_master} -b -m shell -a \
    '{install_root}/venv/bin/pip3 install {install_root}'

## rdmfs_mount.sh の配置

[CS-ood-rdmfs](https://github.com/RCOSDP/CS-ood-rdmfs) は `/usr/local/sbin/rdmfs_mount.sh` を呼び出す前提で作られているため、[CS-rdmfs](https://github.com/RCOSDP/CS-rdmfs) の `bin/start.sh` をその場所に配置します。あわせて、Python 3.11 の virtualenv を利用するように調整します。

In [ ]:
!ansible {target_master} -b -a \
    'install -m 0755 /opt/CS-rdmfs/bin/start.sh /usr/local/sbin/rdmfs_mount.sh'

配置結果を確認します。

In [ ]:
!ansible {target_master} -b -a \
    'ls -l /usr/local/sbin/rdmfs_mount.sh'

In [ ]:
!ansible {target_master} -b -m replace -a \
    'path=/usr/local/sbin/rdmfs_mount.sh regexp=python3 replace={install_root}/venv/bin/python3.11'

## [CS-ood-rdmfs](https://github.com/RCOSDP/CS-ood-rdmfs) の配置

In [ ]:
!ansible {target_master} -b -m file -a \
    'path={ood_sys_root} state=directory mode=0755'

In [ ]:
!ansible {target_master} -b -m git -a \
    'repo={cs_ood_rdmfs_repo} dest={ood_app_root} version={cs_ood_rdmfs_ref} update=yes force=yes'

appディレクトリと `tmp` ディレクトリに必要な権限を付与します。

In [ ]:
!ansible {target_master} -b -m file -a \
    'path={ood_app_root}/tmp state=directory mode=0777'

!ansible {target_master} -b -m file -a \
    'path={ood_app_root} state=directory mode=0777'

Passengerアプリの再読み込みを促し、念のためhttpdをreloadします。

In [ ]:
!ansible {target_master} -b -a \
    'touch /var/www/ood/apps/sys/CS-ood-rdmfs/tmp/restart.txt'

!ansible {target_master} -b -m shell -a \
    'systemctl reload httpd || systemctl restart httpd'

## 配置確認

wrapperとアプリが所定の位置に配置されていることを確認します。

In [ ]:
!ansible {target_master} -b -a \
    'ls -ld /opt/CS-rdmfs /usr/local/sbin/rdmfs_mount.sh /var/www/ood/apps/sys/CS-ood-rdmfs /var/www/ood/apps/sys/CS-ood-rdmfs/tmp'

In [ ]:
!ansible {target_master} -b -a \
    "{install_root}/venv/bin/python3.11 -c 'import rdmfs; print(rdmfs.__file__)'"

!ansible {target_master} -b -m shell -a \
    'test -f /var/www/ood/apps/sys/CS-ood-rdmfs/manifest.yml && echo manifest: OK'

## Open OnDemand での操作

Open OnDemand にログイン後、sys app として追加された [CS-ood-rdmfs](https://github.com/RCOSDP/CS-ood-rdmfs) を開き、`RDM_API_URL`、`RDM_TOKEN`、`RDM_NODE_ID`、`mount_path` を入力して `mount` を実行します。`RDM_NODE_ID` を空にした場合、[CS-rdmfs](https://github.com/RCOSDP/CS-rdmfs) 側の all-projects モードを利用できます。

## 動作確認

### mount状態の確認

mountを実行したユーザとパスを指定します。`check_mount_path` には、Open OnDemand の画面で実際に指定した `mount_path` を設定してください。

In [ ]:
check_mount_user = 'ooduser'
check_mount_path = '/home/ooduser/rdmfs'

マウントテーブルにエントリがあるかを確認します。

In [ ]:
!ansible {target_master} -b -m shell -a \
    "findmnt -T {check_mount_path} || mount | grep ' {check_mount_path} ' || true"

マウント先の内容を、実際に mount を実行したユーザ権限で確認します。

In [ ]:
!ansible {target_master} -b --become-user={check_mount_user} -m shell -a \
    "bash -lc 'ls -la {check_mount_path} | head -20'"

必要に応じて、`rdmfs` 関連プロセスと wrapper の存在も確認します。マウントが見えない場合の切り分けに利用します。

In [ ]:
!ansible {target_master} -b -a \
    "bash -lc 'ps -ef | grep -E \"python3 -m rdmfs|rdmfs_mount\" | grep -v grep || true'"

!ansible {target_master} -b -a \
    'ls -l /usr/local/sbin/rdmfs_mount.sh'